In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"D:\OneDrive\Documents\Downloads\Hospital_Project\treatments_raw.csv")

In [3]:
df.head()

,treatment_id,admission_id,doctor_id,treatment_type,treatment_cost,treatment_date,treatment_status
0,1,27337,62,Surgery,52636.30,2023-01-01,Completed
1,2,982,242,Imaging,7478.12,2023-01-01,Completed
2,3,17082,238,Chemotherapy,27451.58,2023-01-01,Completed
3,4,4096,1,Lab Package,3379.43,2023-01-01,Pending
4,5,26108,227,ICU Care,19993.51,2023-01-01,Completed


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90160 entries, 0 to 90159
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   treatment_id      90160 non-null  int64  
 1   admission_id      90160 non-null  int64  
 2   doctor_id         90160 non-null  int64  
 3   treatment_type    90160 non-null  object 
 4   treatment_cost    89860 non-null  float64
 5   treatment_date    90160 non-null  object 
 6   treatment_status  90160 non-null  object 
dtypes: float64(1), int64(3), object(3)
memory usage: 4.8+ MB


In [5]:
df.isnull().sum()

treatment_id          0
admission_id          0
doctor_id             0
treatment_type        0
treatment_cost      300
treatment_date        0
treatment_status      0
dtype: int64

In [6]:
text_cols = [
    "treatment_type",
    "treatment_status"
]

for col in text_cols:
    df[col] = df[col].str.strip()

In [7]:
len(df)

90160

In [8]:
df.duplicated().sum()

np.int64(160)

In [9]:
df = df.drop_duplicates()

In [10]:
len(df)

90000

In [11]:
df.dtypes

treatment_id          int64
admission_id          int64
doctor_id             int64
treatment_type       object
treatment_cost      float64
treatment_date       object
treatment_status     object
dtype: object

In [12]:
df["treatment_cost"] = pd.to_numeric(
    df["treatment_cost"],
    errors="coerce"
)

In [13]:
df["treatment_date"] = pd.to_datetime(
    df["treatment_date"],
    format="mixed",
    dayfirst=True
)

In [14]:
df.dtypes

treatment_id                 int64
admission_id                 int64
doctor_id                    int64
treatment_type              object
treatment_cost             float64
treatment_date      datetime64[ns]
treatment_status            object
dtype: object

In [15]:
df['cost_outlier_flag'] = 0

In [16]:
df.loc[
    (df['treatment_cost'] < 0) | (df['treatment_cost'] > 500000),
    'cost_outlier_flag'
] = 1

print('Number of cost outliers:', df['cost_outlier_flag'].sum())

Number of cost outliers: 212


In [17]:
# Convert impossible treatment cost into missing value.
df.loc[
    df['cost_outlier_flag'] == 1,
    'treatment_cost'
] = np.nan

In [18]:
# Fill missing treatment cost using median cost of the same treatment type.
df['treatment_cost_clean'] = df.groupby('treatment_type')['treatment_cost'].transform(
    lambda values: values.fillna(values.median())
)

df[['treatment_type', 'treatment_cost', 'treatment_cost_clean', 'cost_outlier_flag']].head()

,treatment_type,treatment_cost,treatment_cost_clean,cost_outlier_flag
0,Surgery,52636.30,52636.30,0
1,Imaging,7478.12,7478.12,0
2,Chemotherapy,27451.58,27451.58,0
3,Lab Package,3379.43,3379.43,0
4,ICU Care,19993.51,19993.51,0


In [19]:
df["treatment_status"].unique()

array(['Completed', 'Pending', 'Cancelled'], dtype=object)

In [20]:
df["treatment_type"] = (
    df["treatment_type"]
    .str.strip()
    .str.title()
)

In [21]:
df.head(10)

,treatment_id,admission_id,doctor_id,treatment_type,treatment_cost,treatment_date,treatment_status,cost_outlier_flag,treatment_cost_clean
0,1,27337,62,Surgery,52636.30,2023-01-01,Completed,0,52636.30
1,2,982,242,Imaging,7478.12,2023-01-01,Completed,0,7478.12
2,3,17082,238,Chemotherapy,27451.58,2023-01-01,Completed,0,27451.58
3,4,4096,1,Lab Package,3379.43,2023-01-01,Pending,0,3379.43
4,5,26108,227,Icu Care,19993.51,2023-01-01,Completed,0,19993.51
5,6,13902,87,Icu Care,25218.80,2023-01-01,Completed,0,25218.80
6,7,15514,94,Dialysis,8080.95,2023-01-01,Completed,0,8080.95
7,8,52,71,Chemotherapy,24527.12,2023-01-01,Completed,0,24527.12
8,9,4911,148,Medication,5935.42,2023-01-01,Completed,0,5935.42
9,10,22064,175,Surgery,118249.75,2023-01-01,Completed,0,118249.75


In [22]:
df.isnull().sum()

treatment_id              0
admission_id              0
doctor_id                 0
treatment_type            0
treatment_cost          512
treatment_date            0
treatment_status          0
cost_outlier_flag         0
treatment_cost_clean      0
dtype: int64

In [23]:
df.to_csv(
    "cleaned_treatments.csv",
    index=False
)